# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitshank-2338/flyrank-ai-starter-notebook/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Can prior search performance help prioritize visible content for editorial review before a subsequent decline in impressions? This is a decision-support ranking, not a recommendation to edit automatically. False positives consume editorial time; the label measures observed change, not whether a refresh would help.

In [ ]:
%pip -q install duckdb
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [userdata.get('HF_TOKEN')])
root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
paths = [f'{root}/month=2026-{m:02d}/*.parquet' for m in (3,4,5,6)]
source = 'read_parquet([' + ','.join(repr(p) for p in paths) + '])'
print('Remote partitions: March–June 2026; only aggregated results will be displayed.')
monthly = con.sql(f"""
SELECT client_hash_id, content_hash_id, date_trunc('month', report_date) AS month,
 COUNT(*) AS days, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
 AVG(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position
FROM {source}
GROUP BY 1,2,3
""").df()
assert not monthly.duplicated(['client_hash_id','content_hash_id','month']).any()
print('Monthly page aggregates:', len(monthly), 'months:', monthly.month.nunique())

def make_pair(past_month, future_month, past_days, future_days):
    past = monthly.loc[(monthly.month == pd.Timestamp(past_month)) & (monthly.days == past_days), ['client_hash_id','content_hash_id','impressions','clicks','position']].copy()
    future = monthly.loc[(monthly.month == pd.Timestamp(future_month)) & (monthly.days == future_days), ['client_hash_id','content_hash_id','impressions']].copy()
    x = past.merge(future, on=['client_hash_id','content_hash_id'], suffixes=('_past','_future'), validate='one_to_one')
    x = x.loc[x.impressions_past >= 100].copy()
    x['ctr'] = x.clicks / x.impressions_past
    x['label'] = (x.impressions_future / future_days < .8 * x.impressions_past / past_days).astype(int)
    return x
train_period = make_pair('2026-03-01','2026-04-01',31,30)
test_period = make_pair('2026-05-01','2026-06-01',31,30)
clients = pd.Series(monthly.client_hash_id.unique())
train_ix, test_ix = next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42).split(clients, groups=clients))
train_clients, test_clients = set(clients.iloc[train_ix]), set(clients.iloc[test_ix])
train = train_period[train_period.client_hash_id.isin(train_clients)].copy()
test = test_period[test_period.client_hash_id.isin(test_clients)].copy()
assert train_clients.isdisjoint(test_clients) and len(train)>0 and len(test)>=50
features = ['impressions_past','clicks','position','ctr']
model = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), RandomForestClassifier(n_estimators=150,min_samples_leaf=20,random_state=42,n_jobs=-1))
model.fit(train[features],train.label)
pred = model.predict_proba(test[features])[:,1]
baseline = test.impressions_past.to_numpy()
def p_at_k(scores, labels, k):
    idx = np.argsort(-np.asarray(scores),kind='stable')[:k]
    return float(np.asarray(labels)[idx].mean())
print('Training pages:',len(train),'held-out pages:',len(test),'training clients:',train.client_hash_id.nunique(),'held-out clients:',test.client_hash_id.nunique())
print('Held-out decline prevalence:',round(test.label.mean(),4))
for name, score in [('Exposure baseline',baseline),('Random forest',pred)]:
    print(name, 'P@20',round(p_at_k(score,test.label,20),4),'P@50',round(p_at_k(score,test.label,50),4),'P@100',round(p_at_k(score,test.label,100),4),'AP',round(average_precision_score(test.label,score),4))
print('Test design: later period and clients not used in training; no future values in features.')

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

FlyRank's gated, pseudonymized daily content-performance warehouse, March–June 2026. The unit is client × content × reporting day, aggregated to client × content × month. Training pairs March with April; evaluation pairs May with June. Only items with complete daily coverage in both windows and at least 100 prior impressions are eligible. Identifiers are used for joins and client separation only; no individual rows are published. Query text, product flags, the fixed query snapshot and future metrics are excluded from predictors. Histories are unbalanced and GA4 coverage varies; engagement is not used here.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
